# SIB - Exercício 1

NumPy array Indexing/Slicing com o dataset iris.

Inclui também exemplos dos métodos `dropna`, `fillna` e `remove_by_index` (exercício 2).

In [1]:
import numpy as np

from si.io.csv_file import read_csv

## 1.1) Carregar o iris.csv

In [2]:
iris = read_csv('../datasets/iris/iris.csv', sep=',', features=True, label=True)

print('dimensões:', iris.shape())
print('features:', iris.features)
print('label:', iris.label)
print('classes:', iris.get_classes())

dimensões: (150, 4)
features: ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']
label: class
classes: ['Iris-setosa' 'Iris-versicolor' 'Iris-virginica']


## 1.2) Penúltima variável independente

Qual a dimensão do array resultante?

In [3]:
penultimate = iris.X[:, -2]

print('feature:', iris.features[-2])
print('dimensão:', penultimate.shape)
print('primeiros 5 valores:', penultimate[:5])

feature: petal_length
dimensão: (150,)
primeiros 5 valores: [1.4 1.4 1.3 1.5 1.4]


**Resposta:** `(150,)`, ou seja, um array unidimensional com 150 valores.

Selecionar uma única coluna com `[:, -2]` elimina a segunda dimensão. Se quisesses manter a forma de matriz `(150, 1)`, usavas `iris.X[:, -2:-1]`.

## 1.3) Últimas 10 amostras

Qual a média de cada variável independente nessas 10 amostras?

In [4]:
last_10 = iris.X[-10:]

print('dimensão:', last_10.shape)
print('médias por feature:')
for feature, mean in zip(iris.features, last_10.mean(axis=0)):
    print(f'  {feature}: {mean:.3f}')

dimensão: (10, 4)
médias por feature:
  sepal_length: 6.450
  sepal_width: 3.030
  petal_length: 5.330
  petal_width: 2.170


**Resposta:** `[6.45, 3.03, 5.33, 2.17]` para sepal_length, sepal_width, petal_length e petal_width.

O `axis=0` calcula a média ao longo das linhas, dando um valor por coluna. Estas 10 amostras são todas da classe Iris-virginica, por isso as médias das pétalas são bastante mais altas do que as do dataset completo.

## 1.4) Amostras com todos os valores menores ou iguais a 6

Quantas amostras se obtêm?

In [5]:
mask = np.all(iris.X <= 6, axis=1)
selected = iris.X[mask]

print('número de amostras:', selected.shape[0])

número de amostras: 89


**Resposta:** 89 amostras.

`iris.X <= 6` devolve uma matriz de booleanos do mesmo tamanho do X. O `np.all(..., axis=1)` colapsa cada linha num único booleano, que só é `True` se **todas** as features dessa amostra cumprirem a condição. Com `np.any` em vez de `np.all` obterias as amostras com *pelo menos uma* feature abaixo de 6, que são muitas mais.

## 1.5) Amostras com classe diferente de 'Iris-setosa'

Quantas amostras se obtêm?

In [6]:
mask = iris.y != 'Iris-setosa'
selected = iris.X[mask]

print('número de amostras:', selected.shape[0])
print('classes presentes:', np.unique(iris.y[mask]))

número de amostras: 100
classes presentes: ['Iris-versicolor' 'Iris-virginica']


**Resposta:** 100 amostras, as 50 de Iris-versicolor mais as 50 de Iris-virginica. O iris tem as três classes perfeitamente equilibradas.

---

# Exercício 2 — exemplos dos métodos novos da classe Dataset

## 2.1) `dropna`

Remove todas as amostras com pelo menos um valor nulo.

In [7]:
iris_na = read_csv('../datasets/iris/iris_missing_data.csv', sep=',', features=True, label=True)

print('antes:', iris_na.shape(), '| nº de NaN:', int(np.isnan(iris_na.X).sum()))

iris_na.dropna()

print('depois:', iris_na.shape(), '| nº de NaN:', int(np.isnan(iris_na.X).sum()))
print('comprimento do y:', len(iris_na.y))

antes: (150, 4) | nº de NaN: 17
depois: (134, 4) | nº de NaN: 0
comprimento do y: 134


De 150 amostras passam a 134. O `y` acompanha a remoção: fica também com 134 entradas, senão as amostras deixariam de corresponder às classes.

## 2.2) `fillna`

Substitui os valores nulos por um valor, pela média ou pela mediana da feature.

In [8]:
# com a média
iris_mean = read_csv('../datasets/iris/iris_missing_data.csv', sep=',', features=True, label=True)
iris_mean.fillna('mean')
print('mean  ->', iris_mean.shape(), '| NaN:', int(np.isnan(iris_mean.X).sum()))

# com a mediana
iris_median = read_csv('../datasets/iris/iris_missing_data.csv', sep=',', features=True, label=True)
iris_median.fillna('median')
print('median->', iris_median.shape(), '| NaN:', int(np.isnan(iris_median.X).sum()))

# com um valor fixo
iris_zero = read_csv('../datasets/iris/iris_missing_data.csv', sep=',', features=True, label=True)
iris_zero.fillna(0.0)
print('0.0   ->', iris_zero.shape(), '| NaN:', int(np.isnan(iris_zero.X).sum()))

mean  -> (150, 4) | NaN: 0
median-> (150, 4) | NaN: 0
0.0   -> (150, 4) | NaN: 0


Ao contrário do `dropna`, o número de amostras não muda: 150 antes e 150 depois. Cada valor em falta é substituído pela estatística **da sua própria coluna**, não por uma estatística global.

## 2.3) `remove_by_index`

Remove uma amostra pelo seu índice.

In [9]:
iris_idx = read_csv('../datasets/iris/iris.csv', sep=',', features=True, label=True)

print('antes:', iris_idx.shape(), '| primeira amostra:', iris_idx.X[0], '| classe:', iris_idx.y[0])

iris_idx.remove_by_index(0)

print('depois:', iris_idx.shape(), '| primeira amostra:', iris_idx.X[0], '| classe:', iris_idx.y[0])
print('comprimento do y:', len(iris_idx.y))

antes: (150, 4) | primeira amostra: [5.1 3.5 1.4 0.2] | classe: Iris-setosa
depois: (149, 4) | primeira amostra: [4.9 3.  1.4 0.2] | classe: Iris-setosa
comprimento do y: 149


In [10]:
# índice fora dos limites
try:
    iris_idx.remove_by_index(1000)
except IndexError as e:
    print('IndexError:', e)

IndexError: Index 1000 is out of bounds for a dataset with 149 samples
